# LSTM from scratch

In [26]:
import os
import re
import numpy as np
from collections import defaultdict, Counter
from nltk.corpus import gutenberg

In [47]:
import nltk

In [48]:
try:
    nltk.data.find("corpora/gutenberg")
except LookupError:
    nltk.download("gutenberg", quiet=True)

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\108pa\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\gutenberg.zip.


True

In [49]:
file_names = gutenberg.fileids()
file_names

['austen-emma.txt',
 'austen-persuasion.txt',
 'austen-sense.txt',
 'bible-kjv.txt',
 'blake-poems.txt',
 'bryant-stories.txt',
 'burgess-busterbrown.txt',
 'carroll-alice.txt',
 'chesterton-ball.txt',
 'chesterton-brown.txt',
 'chesterton-thursday.txt',
 'edgeworth-parents.txt',
 'melville-moby_dick.txt',
 'milton-paradise.txt',
 'shakespeare-caesar.txt',
 'shakespeare-hamlet.txt',
 'shakespeare-macbeth.txt',
 'whitman-leaves.txt']

LSTM Flow

- BPE Tokenizer - tokenize into words/subwords
- LSTMCell      - single-step forward + backward
- LSTM          - unroll the cell over the sequence
- Train         - character / subword language-model training loop

In [54]:
fileids = [
    "austen-emma.txt",
    "austen-sense.txt",
    "austen-persuasion.txt",
    "melville-moby_dick.txt",
    "edgeworth-parents.txt",
]


def clean_gutenberg_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\[[^\]]+\]", " ", text)
    text = re.sub(r"[^a-z0-9.,;:!?\'\"\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_gutenberg_corpus(fileids, max_chars_per_file=None):
    docs = []
    stats = []

    for fileid in fileids:
        raw_text = gutenberg.raw(fileid)
        cleaned = clean_gutenberg_text(raw_text)
        if max_chars_per_file is not None:
            cleaned = cleaned[:max_chars_per_file]

        docs.append(cleaned)
        stats.append({
            "fileid": fileid,
            "chars": len(cleaned),
            "words": len(cleaned.split()),
        })

    corpus = "\n".join(docs)
    return docs, corpus, stats


# Start with a cap while testing. Set max_chars_per_file=None for the full books.
gutenberg_docs, GUTENBERG_CORPUS, gutenberg_stats = load_gutenberg_corpus(
    fileids,
    max_chars_per_file=80_000,
)

print("Loaded files:")
for row in gutenberg_stats:
    print(f"  {row['fileid']}: {row['words']:,} words, {row['chars']:,} chars")

print(f"\nAggregated corpus: {len(GUTENBERG_CORPUS.split()):,} words, {len(GUTENBERG_CORPUS):,} chars")
print(GUTENBERG_CORPUS[:500])

Loaded files:
  austen-emma.txt: 14,516 words, 80,000 chars
  austen-sense.txt: 14,148 words, 80,000 chars
  austen-persuasion.txt: 14,196 words, 80,000 chars
  melville-moby_dick.txt: 14,237 words, 80,000 chars
  edgeworth-parents.txt: 15,071 words, 80,000 chars

Aggregated corpus: 72,168 words, 400,004 chars
volume i chapter i emma woodhouse, handsome, clever, and rich, with a comfortable home and happy disposition, seemed to unite some of the best blessings of existence; and had lived nearly twenty-one years in the world with very little to distress or vex her. she was the youngest of the two daughters of a most affectionate, indulgent father; and had, in consequence of her sister's marriage, been mistress of his house from a very early period. her mother had died too long ago for her to have more 


In [55]:
class BPETokenizer:
    """
    Minimal Byte-Pair Encoding tokeniser.
 
    Training
    --------
    Start with every character as its own token.
    Repeatedly find the most frequent adjacent pair and merge it into a new
    token.  Repeat for `num_merges` steps.
 
    The merge list fully defines the tokeniser; encode() replays the merges
    in order on unseen text.
    """
 
    def __init__(self, num_merges: int = 50):
        self.num_merges = num_merges
        self.merges: list[tuple[str, str]] = []   # ordered merge rules
        self.vocab: dict[str, int] = {}            # token → id
        self.id_to_token: dict[int, str] = {}
 
    # ── helpers ──────────────────────────────
 
    @staticmethod
    def _word_to_chars(word: str) -> list[str]:
        """Split a word into characters; append </w> end-of-word marker."""
        return list(word) + ["</w>"]
 
    @staticmethod
    def _get_pairs(vocab: dict[tuple, int]) -> Counter:
        """Count all adjacent symbol pairs across the vocabulary."""
        pairs: Counter = Counter()
        for symbols, freq in vocab.items():
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i + 1])] += freq
        return pairs
 
    @staticmethod
    def _merge_pair(pair: tuple[str, str], vocab: dict[tuple, int]) -> dict[tuple, int]:
        """Replace every occurrence of `pair` with the merged token."""
        new_vocab: dict[tuple, int] = {}
        bigram = " ".join(pair)                    # e.g. "a b"
        replacement = "".join(pair)                # e.g. "ab"
        for symbols, freq in vocab.items():
            # re-join to string, replace, split back
            joined = " ".join(symbols)
            merged = joined.replace(bigram, replacement)
            new_vocab[tuple(merged.split())] = freq
        return new_vocab
 
    # ── public API ───────────────────────────
 
    def train(self, text: str) -> None:
        """Learn BPE merges from `text`."""
        # build initial vocab: word → frequency, word split into chars
        word_freq: Counter = Counter(re.findall(r'\S+', text.lower()))
        vocab: dict[tuple, int] = {
            tuple(self._word_to_chars(w)): f
            for w, f in word_freq.items()
        }
 
        # learn merges
        for _ in range(self.num_merges):
            pairs = self._get_pairs(vocab)
            if not pairs:
                break
            best = max(pairs, key=pairs.__getitem__)
            vocab = self._merge_pair(best, vocab)
            self.merges.append(best)
 
        # build final vocabulary from all tokens that appear after merging
        all_tokens: set[str] = set()
        for symbols in vocab:
            all_tokens.update(symbols)
        all_tokens.add("<unk>")
 
        self.vocab = {tok: i for i, tok in enumerate(sorted(all_tokens))}
        self.id_to_token = {i: tok for tok, i in self.vocab.items()}
 
    def _tokenise_word(self, word: str) -> list[str]:
        """Apply learned merges to a single word."""
        symbols = self._word_to_chars(word)
        for pair in self.merges:
            bigram = " ".join(pair)
            replacement = "".join(pair)
            while True:
                joined = " ".join(symbols)
                if bigram not in joined:
                    break
                symbols = joined.replace(bigram, replacement, 1).split()
        return symbols
 
    def encode(self, text: str) -> list[int]:
        """Text → list of token ids."""
        ids: list[int] = []
        for word in re.findall(r'\S+', text.lower()):
            for tok in self._tokenise_word(word):
                ids.append(self.vocab.get(tok, self.vocab["<unk>"]))
        return ids
 
    def decode(self, ids: list[int]) -> str:
        """Token ids → text (removes </w> markers)."""
        tokens = [self.id_to_token.get(i, "<unk>") for i in ids]
        return " ".join("".join(tokens).split("</w>")).strip()
 
    @property
    def vocab_size(self) -> int:
        return len(self.vocab)

In [56]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def sigmoid_grad(s: np.ndarray) -> np.ndarray:
    return s * (1.0 - s)

def tanh_grad(t: np.ndarray) -> np.ndarray:
    return 1.0 - t ** 2

class LSTMCell:
    """
    input_size: x_t
    hidden_size: h_t
    W shape: (4.H, H+I)     -> rows: [f, i, o, g]
    b shape: (4.H, )

    Forward pass equations
    ----------------------
        z      = W · [h_{t-1}; x_t] + b        concat then linear
        f_t    = σ(z[0:H])                      forget gate
        i_t    = σ(z[H:2H])                     input gate
        o_t    = σ(z[2H:3H])                    output gate
        g_t    = tanh(z[3H:4H])                 candidate cell
        c_t    = f_t ⊙ c_{t-1} + i_t ⊙ g_t    cell state
        h_t    = o_t ⊙ tanh(c_t)               hidden state
    """

    def __init__(self, input_size: int, hidden_size: int):
        self.I = input_size
        self.H = hidden_size

        k = np.sqrt(1.0 / (input_size + hidden_size))
        self.W = np.random.uniform(-k, k, (4*hidden_size, hidden_size + input_size))
        self.b = np.zeros(4*hidden_size)

        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b) 

    
    def forward(self, x: np.ndarray, h_prev: np.ndarray, c_prev: np.ndarray) -> tuple[np.ndarray, np.ndarray, dict]:
        ### returns h_t, c_t, and a cache dict for backward
        H = self.H
        hx = np.concatenate([h_prev, x])
        z = self.W @ hx + self.b 

        f = sigmoid(z[0*H : 1*H])
        i = sigmoid(z[1*H : 2*H])
        o = sigmoid(z[2*H : 3*H])
        g = np.tanh(z[3*H : 4*H])

        c = f * c_prev + i * g # new cell state
        tanh_c = np.tanh(c)
        h = o * tanh_c 

        cache = dict(hx=hx, f=f, i=i, o=o, g=g, c=c, tanh_c=tanh_c, c_prev=c_prev)

        return h, c, cache
    

    def backward(self, dh: np.ndarray, dc: np.ndarray, cache: dict) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        ### BPTT through a single cell step

        ### returns dx, dh_prev, dc_prev
        ### also accumulates dW and db 

        H = self.H 
        f, i, o, g = cache['f'], cache['i'], cache['o'], cache['g']
        c, tanh_c, c_prev, hx = cache['c'], cache['tanh_c'], cache['c_prev'], cache['hx']

        # ∂L/∂o   — from dh
        do = dh * tanh_c                               # (H,)
        # ∂L/∂tanh(c)
        dtanh_c = dh * o                               # (H,)
        # ∂L/∂c   — from both dtanh_c and incoming dc
        dc_full = dtanh_c * tanh_grad(tanh_c) + dc    # (H,)
 
        # ∂L/∂f, ∂L/∂i, ∂L/∂g, ∂L/∂c_prev
        df     = dc_full * c_prev                      # (H,)
        di     = dc_full * g                           # (H,)
        dg     = dc_full * i                           # (H,)
        dc_prev = dc_full * f                          # (H,) — flows back in time
 
        # ∂L/∂pre-activations (before sigmoid / tanh)
        dz_f = df * sigmoid_grad(f)                    # (H,)
        dz_i = di * sigmoid_grad(i)                    # (H,)
        dz_o = do * sigmoid_grad(o)                    # (H,)
        dz_g = dg * tanh_grad(g)                       # (H,)
 
        dz = np.concatenate([dz_f, dz_i, dz_o, dz_g]) # (4H,)
 
        # accumulate parameter gradients
        self.dW += np.outer(dz, hx)                    # (4H, H+I)
        self.db += dz                                  # (4H,)
 
        # gradient w.r.t. [h_prev; x]
        dhx = self.W.T @ dz                            # (H+I,)
        dh_prev = dhx[:H]
        dx      = dhx[H:]
 
        return dx, dh_prev, dc_prev
 
    def zero_grad(self) -> None:
        self.dW[:] = 0.0
        self.db[:] = 0.0

In [57]:
class LSTM:
    def __init__(self, vocab_size: int, embed_dim: int, hidden_size: int):
        self.V = vocab_size
        self.E = embed_dim
        self.H = hidden_size

        # embedding
        self.embed = np.random.randn(vocab_size, embed_dim) * 0.01

        # LSTM cell
        self.cell = LSTMCell(embed_dim, hidden_size)

        # output projection
        self.W_out = np.random.randn(vocab_size, hidden_size) * 0.01
        self.b_out = np.zeros(vocab_size)

        # gradient accumulators for embedding and output layerr
        self.d_embed = np.zeros_like(self.embed)
        self.dW_out  = np.zeros_like(self.W_out)
        self.db_out  = np.zeros_like(self.b_out)


    def _softmax(self, logits: np.ndarray) -> np.ndarray:
        e = np.exp(logits - logits.max())
        return e / e.sum()
    

    def forward(
            self,
            token_ids: list[int],
            h0: np.ndarray | None = None,
            c0: np.ndarray | None = None,
    ) -> tuple[list[np.ndarray], list[np.ndarray], list[dict], list[np.ndarray]]:
        
        """
        Forward pass over a full sequence

        Returns
        ______
        hs      : list of h_t for each step (T * H)
        cs      : list of c_t for each step (T * H)
        caches  : list of per-step caches   (for backward)
        logits  : list of raw output logits (T * V)
        """

        T = len(token_ids)
        h = np.zeros(self.H) if h0 is None else h0
        c = np.zeros(self.H) if c0 is None else c0 

        hs, cs, caches, logits = [], [], [], []
        self._inputs = token_ids 

        for t in range(T):
            x = self.embed[token_ids[t]]
            h, c, cache = self.cell.forward(x, h, c)
            logit = self.W_out @ h + self.b_out 
            hs.append(h)
            cs.append(c)
            caches.append(cache)
            logits.append(logit)

        return hs, cs, caches, logits
    

    def cross_entropy_loss(self, logits: list[np.ndarray], targets: list[int]) -> tuple[float, list[np.ndarray]]:
        ### compute mean cross-entropy loss and per-step gradient

        T = len(targets)
        loss = 0.0
        d_logits = []
        for t in range(T):
            probs = self._softmax(logits[t])
            loss -= np.log(probs[targets[t]] + 1e-9)

            dl = probs.copy()
            dl[targets[t]] -= 1.0
            d_logits.append(dl / T) # divide by T for mean loss

        return loss / T, d_logits
    

    def backward(self, hs: list[np.ndarray], caches: list[dict], d_logits: list[np.ndarray]) -> None:
        T = len(hs)
        dh_next = np.zeros(self.H)
        dc_next = np.zeros(self.H)

        self.cell.zero_grad()
        self.dW_out[:] = 0.0
        self.db_out[:] = 0.0
        self.d_embed[:] = 0.0

        for t in reversed(range(T)):
            dl = d_logits[t] # get the change in gradident

            # output projection  gradients
            self.dW_out += np.outer(dl, hs[t])
            self.db_out += dl 

            # gradient flowing into h_t from output layer
            dh = self.W_out.T @ dl + dh_next 
            
            dx, dh_next, dc_next = self.cell.backward(dh, dc_next, caches[t])

            # embedding gradient
            self.d_embed[self._inputs[t]] += dx


    def update(self, lr: float = 1e-3, clip: float = 5.0) -> None:
        ### SGD with gradient clipping 
        for param, grad in [
            (self.cell.W,   self.cell.dW),
            (self.cell.b,   self.cell.db),
            (self.W_out,    self.dW_out),
            (self.b_out,    self.db_out),
            (self.embed,    self.d_embed),
        ]:
            np.clip(grad, -clip, clip, out=grad)
            param -= lr * grad


    # ── sampling ─────────────────────────────

    def sample(
        self,
        seed_ids: list[int],
        length: int = 40,
        temperature: float = 1.0,
    ) -> list[int]:
        """Autoregressive sampling from the model."""
        h = np.zeros(self.H)
        c = np.zeros(self.H)
        generated = list(seed_ids)
 
        # warm up on seed
        for tid in seed_ids:
            x = self.embed[tid]
            h, c, _ = self.cell.forward(x, h, c)
 
        # sample
        for _ in range(length):
            logit = self.W_out @ h + self.b_out
            probs = self._softmax(logit / temperature)
            next_id = int(np.random.choice(len(probs), p=probs))
            generated.append(next_id)
            x = self.embed[next_id]
            h, c, _ = self.cell.forward(x, h, c)
 
        return generated


In [58]:
def train(
        text: str,
        num_merges: int = 80,
        embed_dim: int = 32,
        hidden_size: int = 64,
        seq_len: int = 20,
        epochs: int = 200,
        lr: float = 5e-3):
    
    ### tokenizer
    print("\n[1] Training BPE Tokenizer")
    tok = BPETokenizer(num_merges = num_merges)
    tok.train(text)
    ids = tok.encode(text)
    V   = tok.vocab_size

    print(f"    vocab size : {V}")
    print(f"    tokens     : {len(ids)}")
    print(f"    merges     : {num_merges}")
    print(f"    first 10 tokens: {[tok.id_to_token[i] for i in ids[:10]]}")
 
    if len(ids) < seq_len + 1:
        raise ValueError("Text too short for the given seq_len.")
    

    ### model 
    print("\n[2] Building LSTM model")
    model = LSTM(V, embed_dim, hidden_size)
    total_params = (
        model.embed.size + model.cell.W.size + model.cell.b.size + model.W_out.size + model.b_out.size
    )

    print(f"\ttotal parameters: {total_params}")


    ### training loop
    losses = []
    for epoch in range(1, epochs + 1):
        start = np.random.randint(0, len(ids) - seq_len - 1) # random weights 
        inputs = ids[start: start+seq_len]
        targets = ids[start+1: start+seq_len+1]

        
        hs, cs, caches, logits = model.forward(inputs)
        loss, d_logits = model.cross_entropy_loss(logits, targets)
        model.backward(hs, caches, d_logits)
        model.update(lr=lr)
 
        losses.append(loss)
        if epoch % 20 == 0 or epoch == 1:
            print(f"  epoch {epoch:4d}  loss={loss:.4f}  perplexity={np.exp(loss):.2f}")
        
    # ── sample ───────────────────────────────
    print("\n[4] Sampling from the trained model …\n")
    seed = ids[:3]
    seed_text = tok.decode(seed)
    sampled = model.sample(seed, length=50, temperature=0.8)
    sampled_text = tok.decode(sampled)
    print(f"  seed    : {seed_text!r}")
    print(f"  sampled : {sampled_text!r}")
    print("\nDone.")
 
    return model, tok, losses
 

In [59]:
# Optional: train the from-scratch NumPy LSTM.
# The PyTorch training section below is the main path now.
#
# model, tokeniser, losses = train(
#     text=GUTENBERG_CORPUS,
#     num_merges=80,
#     embed_dim=32,
#     hidden_size=64,
#     seq_len=24,
#     epochs=300,
#     lr=5e-3,
# )


[1] Training BPE Tokenizer
    vocab size : 515
    tokens     : 212881
    merges     : 80
    first 10 tokens: ['v', 'o', 'l', 'u', 'm', 'e</w>', 'i</w>', 'cha', 'p', 't']

[2] Building LSTM model
	total parameters: 74787
  epoch    1  loss=6.2442  perplexity=515.00
  epoch   20  loss=6.2426  perplexity=514.21
  epoch   40  loss=6.2413  perplexity=513.53
  epoch   60  loss=6.2410  perplexity=513.37
  epoch   80  loss=6.2417  perplexity=513.76
  epoch  100  loss=6.2377  perplexity=511.70
  epoch  120  loss=6.2383  perplexity=512.01
  epoch  140  loss=6.2356  perplexity=510.61
  epoch  160  loss=6.2347  perplexity=510.14
  epoch  180  loss=6.2328  perplexity=509.20
  epoch  200  loss=6.2345  perplexity=510.06
  epoch  220  loss=6.2321  perplexity=508.83
  epoch  240  loss=6.2280  perplexity=506.74
  epoch  260  loss=6.2266  perplexity=506.02
  epoch  280  loss=6.2273  perplexity=506.38
  epoch  300  loss=6.2238  perplexity=504.60

[4] Sampling from the trained model …

  seed    : 'vo

In [62]:
# Optional: sample from the NumPy LSTM after running the optional cell above.
#
# seed_text = "the wind blows in my face like the "
# seed_ids = tokeniser.encode(seed_text)
# sampled_ids = model.sample(seed_ids, length=10, temperature=0.8)
# print(tokeniser.decode(sampled_ids))

the wind blows in my face like the hatallanore bether aller aritien oorlin


## Using Pytorch

In [63]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class TokenSequenceDataset(Dataset):
    def __init__(self, token_ids, seq_len=64, stride=None, max_sequences=None):
        self.ids = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len
        stride = stride or seq_len

        max_start = len(self.ids) - seq_len - 1
        if max_start < 1:
            raise ValueError("Not enough tokens for the selected seq_len")

        starts = list(range(0, max_start, stride))
        if max_sequences is not None and len(starts) > max_sequences:
            picks = np.linspace(0, len(starts) - 1, max_sequences, dtype=int)
            starts = [starts[i] for i in picks]

        self.starts = starts

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        start = self.starts[idx]
        x = self.ids[start:start + self.seq_len]
        y = self.ids[start + 1:start + self.seq_len + 1]
        return x, y


class TorchLSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_size=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        out, hidden = self.lstm(x, hidden)
        logits = self.output(out)
        return logits, hidden

In [64]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


cuda


In [65]:

def train_torch_lstm(
    text,
    num_merges=120,
    seq_len=64,
    embed_dim=128,
    hidden_size=256,
    num_layers=2,
    dropout=0.2,
    batch_size=64,
    epochs=5,
    lr=1e-3,
    max_tokens=200_000,
    max_sequences=20_000,
    device=None,
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    print("[1] Training BPE tokenizer on cleaned Gutenberg corpus")
    tokenizer = BPETokenizer(num_merges=num_merges)
    tokenizer.train(text)
    token_ids = tokenizer.encode(text)
    if max_tokens is not None:
        token_ids = token_ids[:max_tokens]

    print(f"    vocab size : {tokenizer.vocab_size}")
    print(f"    tokens     : {len(token_ids):,}")
    print(f"    device     : {device}")

    dataset = TokenSequenceDataset(
        token_ids,
        seq_len=seq_len,
        stride=seq_len,
        max_sequences=max_sequences,
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    model = TorchLSTMLanguageModel(
        vocab_size=tokenizer.vocab_size,
        embed_dim=embed_dim,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    print("\n[2] Training PyTorch LSTM")
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_batches = 0

        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits, _ = model(x)
            loss = criterion(logits.reshape(-1, tokenizer.vocab_size), y.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            total_batches += 1

        avg_loss = total_loss / max(total_batches, 1)
        losses.append(avg_loss)
        print(f"  epoch {epoch:3d}/{epochs} loss={avg_loss:.4f} perplexity={np.exp(avg_loss):.2f}")

    return model, tokenizer, losses, device


@torch.no_grad()
def sample_torch_lstm(model, tokenizer, seed_text, length=80, temperature=0.8, device=None):
    device = device or next(model.parameters()).device
    model.eval()

    generated = tokenizer.encode(seed_text)
    if not generated:
        generated = [tokenizer.vocab["<unk>"]]

    for _ in range(length):
        x = torch.tensor([generated], dtype=torch.long, device=device)
        logits, _ = model(x)
        next_logits = logits[0, -1] / max(temperature, 1e-6)
        probs = torch.softmax(next_logits, dim=-1)
        next_id = int(torch.multinomial(probs, num_samples=1).item())
        generated.append(next_id)

    return tokenizer.decode(generated)

In [68]:
torch_model, torch_tokenizer, torch_losses, torch_device = train_torch_lstm(
    GUTENBERG_CORPUS,
    num_merges=120,
    seq_len=64,
    embed_dim=128,
    hidden_size=256,
    num_layers=2,
    batch_size=64,
    epochs=10,
    lr=1e-3,
    max_tokens=200_000,
    max_sequences=20_000,
)

[1] Training BPE tokenizer on cleaned Gutenberg corpus
    vocab size : 819
    tokens     : 186,348
    device     : cuda

[2] Training PyTorch LSTM
  epoch   1/10 loss=5.4642 perplexity=236.09
  epoch   2/10 loss=5.1600 perplexity=174.16
  epoch   3/10 loss=5.1092 perplexity=165.54
  epoch   4/10 loss=4.8770 perplexity=131.24
  epoch   5/10 loss=4.6127 perplexity=100.76
  epoch   6/10 loss=4.3967 perplexity=81.18
  epoch   7/10 loss=4.2074 perplexity=67.18
  epoch   8/10 loss=4.0503 perplexity=57.41
  epoch   9/10 loss=3.9269 perplexity=50.75
  epoch  10/10 loss=3.8261 perplexity=45.88


In [73]:
sentence = "there was a beautiful looking woman sitting by the "


print(sample_torch_lstm(
    torch_model,
    torch_tokenizer,
    seed_text=sentence,
    length=20,
    temperature=0.8,
    device=torch_device,
))

there was a beautiful looking woman sitting by the dest partsand what for wimpe, with wthed the could
